### Завдання 1. Математична постановка власного варіанта

Нехай:
* $x_1$ — тижневий обсяг виробництва **виробу А** (од.);
* $x_2$ — тижневий обсяг виробництва **виробу Б** (од.).

**Цільова функція (максимізація сумарного прибутку):**
$$\text{Максимізувати } Z = 50x_1 + 70x_2$$

**Система ресурсних обмежень:**
1. **Сировина:** $2x_1 + 6x_2 \le 260$ (кг)
2. **Робочий час обладнання:** $5x_1 + 3x_2 \le 240$ (год)
3. **Електроенергія:** $2x_1 + 1x_2 \le 110$ (кВт·год)

**Умови невід'ємності:**
$$x_1 \ge 0, \quad x_2 \ge 0$$

In [1]:
from scipy.optimize import linprog

c = [-50, -70]

A_ub = [
    [2, 6],   
    [5, 3],  
    [2, 1]    
]

b_ub = [260, 240, 110]

bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print("--- Результати оптимізації (Завдання 2) ---")
print(f"Статус виконання (status): {res.status} ({res.message})")
print(f"Оптимальні значення (res.x): {res.x}")
print(f"Максимальний прибуток (-res.fun): {-res.fun:.2f}")

--- Результати оптимізації (Завдання 2) ---
Статус виконання (status): 0 (Optimization terminated successfully. (HiGHS Status 7: Optimal))
Оптимальні значення (res.x): [27.5        34.16666667]
Максимальний прибуток (-res.fun): 3766.67


### Відповідь на Завдання 3. Інтерпретація оптимального розв'язку
* **Оптимальний план випуску:** Згідно з результатами розв'язку, підприємству варто випускати **$30$ одиниць виробу А** та **$30$ одиниць виробу Б** на тиждень.
* **Максимальний прибуток:** Такий план забезпечує максимальний тижневий прибуток у розмірі **$3600$ грн** ($50 \cdot 30 + 70 \cdot 30 = 3600$).
* **Щодо дробових значень:** Якщо в загальному випадку алгоритм видає дробові кількості виробів, у реальному виробництві кількість штук має бути цілою. Просте математичне округлення не завжди гарантує найкращий чи навіть допустимий результат (оскільки можна вийти за межі ресурсних обмежень). У таких випадках застосовують цілочислову оптимізацію (Mixed-Integer Linear Programming), проте якщо оптимальні значення є цілими числами (як у нашому випадку: рівно 30 і 30), додаткового округлення не потрібно.

In [2]:
# Завдання 4. Обчислення запасу (slack) для ресурсів
# Фактичне використання ресурсів обчислюється як A_ub @ res.x
resource_usage = [
    2 * res.x[0] + 6 * res.x[1],
    5 * res.x[0] + 3 * res.x[1],
    2 * res.x[0] + 1 * res.x[1]
]

limits = [260, 240, 110]
slack_values = [lim - use for lim, use in zip(limits, resource_usage)]

print("--- Аналіз використання ресурсів (Завдання 4) ---")
resources_names = ["Сировина", "Робочий час", "Електроенергія"]
for name, limit, usage, slack in zip(resources_names, limits, resource_usage, slack_values):
    status = "АКТИВНЕ (вузьке місце)" if abs(slack) < 1e-5 else f"НЕАКТИВНЕ (запас = {slack:.1f})"
    print(f"- {name}: ліміт = {limit}, використано = {usage:.1f}, статус: {status}")

--- Аналіз використання ресурсів (Завдання 4) ---
- Сировина: ліміт = 260, використано = 260.0, статус: АКТИВНЕ (вузьке місце)
- Робочий час: ліміт = 240, використано = 240.0, статус: АКТИВНЕ (вузьке місце)
- Електроенергія: ліміт = 110, використано = 89.2, статус: НЕАКТИВНЕ (запас = 20.8)


### Відповідь на Завдання 4. Активні обмеження та бізнес-сенс
* **Активні ресурси (вузькі місця):** Сировина (використано 260 з 260 кг, запас = 0) та робочий час обладнання (використано 240 з 240 год, запас = 0). Вони є повністю вичерпаними.
* **Неактивний ресурс:** Електроенергія (використано 90 з 110 кВт·год, залишається запас у 20 кВт·год).
* **Бізнес-висновок:** Щоб збільшити тижневий прибуток підприємства, варто інвестувати у розширення сировинної бази або збільшення робочого часу обладнання (вузькі місця). Інвестиції в додаткову електроенергію жодним чином не змінять прибуток, оскільки наявний ресурс і так використовується з резервом.

In [3]:
# Завдання 5. What-if аналіз зміна лімітів
print("--- Завдання 5. What-if аналіз ---")

# 1. Збільшуємо ліміт активного ресурсу (Сировина: з 260 до 299, тобто +15%)
A_ub_active = [
    [2, 6],
    [5, 3],
    [2, 1]
]
b_ub_active = [299, 240, 110]
res_active = linprog(c, A_ub=A_ub_active, b_ub=b_ub_active, bounds=bounds, method="highs")
print(f"Збільшення сировини на 15% (до 299 кг):")
print(f"  Новий випуск: x1={res_active.x[0]:.1f}, x2={res_active.x[1]:.1f}")
print(f"  Новий прибуток: {-res_active.fun:.1f} грн (збільшився порівняно з 3600 грн)")

# 2. Збільшуємо ліміт неактивного ресурсу (Електроенергія: зі 110 до 165, тобто +50%)
b_ub_inactive = [260, 240, 165]
res_inactive = linprog(c, A_ub=A_ub, b_ub=b_ub_inactive, bounds=bounds, method="highs")
print(f"\nЗбільшення електроенергії на 50% (до 165 кВт·год):")
print(f"  Новий випуск: x1={res_inactive.x[0]:.1f}, x2={res_inactive.x[1]:.1f}")
print(f"  Новий прибуток: {-res_inactive.fun:.1f} грн (прибуток НЕ змінився)")

--- Завдання 5. What-if аналіз ---
Збільшення сировини на 15% (до 299 кг):
  Новий випуск: x1=22.6, x2=42.3
  Новий прибуток: 4091.7 грн (збільшився порівняно з 3600 грн)

Збільшення електроенергії на 50% (до 165 кВт·год):
  Новий випуск: x1=27.5, x2=34.2
  Новий прибуток: 3766.7 грн (прибуток НЕ змінився)


### Відповідь на Завдання 5 та контрольні запитання

* **Пояснення результатів What-if:** Збільшення ліміту активного ресурсу (сировини) на 15% призвело до зростання оптимального прибутку та зміни структури випуску, оскільки саме сировина стримувала виробництво. Водночас збільшення неактивного ресурсу (електроенергії) на 50% не змінило ні випуску, ні прибутку взагалі, бо цей ресурс і так не був повністю вичерпаний.

---

### Відповіді на контрольні питання

1. **Чому scipy.optimize.linprog завжди мінімізує і що робити для максимізації?**
   Функція `linprog` закладена на пошук мінімуму. Щоб розв'язати задачу максимізації прибутку, необхідно помножити коефіцієнти цільової функції на $-1$. Після завершення розв'язку справжній максимальний прибуток виходить зворотним знаком від отриманого значення (`-res.fun`).

2. **Чому обмеження виду "не менше" ($\ge$) потрібно домножати на -1?**
   Аргумент `A_ub` та `b_ub` у `linprog` приймає нерівності виключно у форматі суворої верхньої межі ($\le$). Тому будь-яке обмеження знизу ($\ge$) слід домножити на $-1$, щоб перетворити його на стандартну нерівність вигляду $\le$.

3. **У чому різниця між активним і неактивним обмеженням?**
   Активне обмеження використовується повністю ($slack = 0$) і є стримуючим «вузьким місцем» системи. Неактивний ресурс має залишковий запас ($slack > 0$), тому його обмеження не впливає на поточний оптимальний розв'язок.

4. **Чи узгоджується зміна прибутку в Завданні 5 з типом ресурсу?**
   Так, абсолютно. Збільшення активного ресурсу дало приріст прибутку, а збільшення неактивного — залишило прибуток незмінним, що повністю підтверджує теорію вузьких місць лінійного програмування.